# Voice Dubbing Pipeline

This notebook transcribes, translates, and dubs videos from English to Hindi.

## Features
- Extracts audio from video
- Transcribes English speech using Whisper
- Translates to Hindi using Google Translate
- Generates Hindi speech using Edge TTS
- Preserves original timing and pauses
- Keeps original audio for counting segments

## 1. Install Dependencies

In [ ]:
# Install required packages
!apt-get update && apt-get install -y ffmpeg
!pip install yt-dlp ffmpeg-python openai-whisper deep-translator edge-tts numpy scipy tqdm librosa soundfile

## 2. Setup Project

In [ ]:
import os
import shutil
from pathlib import Path

# Create project structure
PROJECT_DIR = Path("/content/voice_dubbing")
PROJECT_DIR.mkdir(exist_ok=True)

for d in ["videos", "audio", "outputs"]:
    (PROJECT_DIR / d).mkdir(exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")

## 3. Create Pipeline Files

In [ ]:
# Create config.py
config_content = '''
import os
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent
VIDEOS_DIR = BASE_DIR / "videos"
AUDIO_DIR = BASE_DIR / "audio"
OUTPUTS_DIR = BASE_DIR / "outputs"

WHISPER_MODEL = "tiny"
TARGET_LANGUAGE = "hi"
SOURCE_LANGUAGE = "en"

def setup_directories():
    for directory in [VIDEOS_DIR, AUDIO_DIR, OUTPUTS_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
'''

with open(PROJECT_DIR / "config.py", "w") as f:
    f.write(config_content)

# Create utils.py
utils_content = '''
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

def initialize_project_environment():
    from config import setup_directories
    setup_directories()
    logger.info("Project directories ready.")
'''

with open(PROJECT_DIR / "utils.py", "w") as f:
    f.write(utils_content)

print("Created config.py and utils.py")

In [ ]:
# Create audio_processing.py
audio_proc_content = '''
import os
import ffmpeg
from config import AUDIO_DIR
from utils import logger

def extract_audio(video_path: str) -> str:
    logger.info(f"Extracting audio from video: {video_path}")
    
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video file not found: {video_path}")
        
    filename = os.path.basename(video_path)
    name_without_ext = os.path.splitext(filename)[0]
    audio_output_path = os.path.join(AUDIO_DIR, f"{name_without_ext}.wav")
    
    (
        ffmpeg
        .input(video_path)
        .output(audio_output_path, ac=1, ar="16000", format="wav")
        .overwrite_output()
        .run(quiet=True, capture_stdout=True, capture_stderr=True)
    )
    logger.info(f"Audio extracted successfully to: {audio_output_path}")
    return audio_output_path
'''

with open(PROJECT_DIR / "audio_processing.py", "w") as f:
    f.write(audio_proc_content)

print("Created audio_processing.py")

In [ ]:
# Create transcription.py
transcribe_content = '''
import whisper
from utils import logger
from config import WHISPER_MODEL

def transcribe_audio(audio_path: str) -> list:
    logger.info(f"Loading Whisper model '{WHISPER_MODEL}' for transcription...")
    model = whisper.load_model(WHISPER_MODEL)
    
    logger.info(f"Transcribing audio: {audio_path}")
    result = model.transcribe(audio_path, language="en")
    
    segments = []
    for seg in result["segments"]:
        segments.append({
            "text": seg["text"].strip(),
            "start": seg["start"],
            "end": seg["end"]
        })
    
    logger.info(f"Transcription complete. {len(segments)} segments found.")
    return segments
'''

with open(PROJECT_DIR / "transcription.py", "w") as f:
    f.write(transcribe_content)

print("Created transcription.py")

In [ ]:
# Create translation.py
trans_content = '''
from deep_translator import GoogleTranslator
from utils import logger
from config import SOURCE_LANGUAGE, TARGET_LANGUAGE
import time
import copy
import re

def is_counting_text(text: str) -> bool:
    text_lower = text.lower().strip()
    counting_words = [
        "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
        "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen", "twenty",
        "1", "2", "3", "4", "5", "6", "7", "8", "9", "10",
        "first", "second", "third", "fourth", "fifth"
    ]
    text_clean = re.sub(r"[,!.?\-;:]", " ", text_lower)
    words = re.findall(r"\\b\\w+\\b", text_clean)
    if not words:
        return False
    counting_matches = sum(1 for w in words if w in counting_words)
    if len(words) <= 3 and counting_matches >= 1:
        return True
    return counting_matches >= len(words) * 0.3

def translate_segments(segments: list, progress_callback=None) -> list:
    logger.info(f"Translating {len(segments)} segments from {SOURCE_LANGUAGE} to {TARGET_LANGUAGE}...")
    translator = GoogleTranslator(source=SOURCE_LANGUAGE, target=TARGET_LANGUAGE)
    translated_segments = []
    
    for i, segment in enumerate(segments):
        original_text = segment["text"].strip()
        keep_original = is_counting_text(original_text)
        
        if not original_text:
            translated_text = ""
        elif keep_original:
            logger.info(f"Segment {i}: Counting detected - will keep original audio")
            translated_text = original_text
        else:
            try:
                translated_text = translator.translate(original_text)
                time.sleep(0.3)
            except Exception as e:
                logger.warning(f"Translation failed: {e}")
                translated_text = original_text
        
        new_segment = copy.deepcopy(segment)
        new_segment["text"] = translated_text
        new_segment["keep_original"] = keep_original
        translated_segments.append(new_segment)
    
    logger.info("Translation complete.")
    return translated_segments
'''

with open(PROJECT_DIR / "translation.py", "w") as f:
    f.write(trans_content)

print("Created translation.py")

In [ ]:
# Create tts_generation.py
tts_content = '''
import os
import asyncio
import tempfile
import numpy as np
import soundfile as sf
from config import AUDIO_DIR, TARGET_LANGUAGE
from utils import logger
import librosa

def detect_speech_regions(audio_path: str, sample_rate: int = 16000) -> list:
    logger.info("Detecting speech regions in original audio...")
    y, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
    energy = np.abs(librosa.stft(y))
    energy_db = librosa.amplitude_to_db(energy, ref=np.max)
    energy_per_frame = energy_db.mean(axis=0)
    silence_threshold = -50
    frame_length = 2048
    hop_length = 512
    frame_duration = hop_length / sr
    is_speech = energy_per_frame > silence_threshold
    regions = []
    current_speech = is_speech[0]
    start_frame = 0
    for i in range(1, len(is_speech)):
        if is_speech[i] != current_speech:
            regions.append({
                "speech": bool(current_speech),
                "start": float(start_frame * frame_duration),
                "end": float(i * frame_duration)
            })
            current_speech = is_speech[i]
            start_frame = i
    regions.append({
        "speech": bool(current_speech),
        "start": float(start_frame * frame_duration),
        "end": float(len(is_speech) * frame_duration)
    })
    merged_regions = []
    for r in regions:
        if r["speech"] and merged_regions and not merged_regions[-1]["speech"]:
            gap = r["start"] - merged_regions[-1]["end"]
            if gap < 0.3:
                merged_regions[-1]["end"] = r["end"]
                continue
        merged_regions.append(r)
    logger.info(f"Detected {len(merged_regions)} audio regions")
    return merged_regions

VOICE_MAP = {
    "hi": "hi-IN-MadhurNeural",
    "en": "en-US-GuyNeural",
}

async def _synthesize_segment(text: str, voice: str, output_path: str):
    import edge_tts
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_path)

def generate_dubbed_audio(segments: list, original_audio_path: str,
                           video_duration_sec: float, progress_callback=None) -> tuple[str, str]:
    voice = VOICE_MAP.get(TARGET_LANGUAGE, "hi-IN-MadhurNeural")
    logger.info(f"Using edge-tts voice: {voice}")
    speech_regions = detect_speech_regions(original_audio_path)
    original_audio, orig_sr = librosa.load(original_audio_path, sr=24000, mono=True)
    sample_rate = 24000
    total_samples = int((video_duration_sec + 30.0) * sample_rate)
    final_audio = np.zeros(total_samples, dtype=np.float32)
    filename = os.path.basename(original_audio_path)
    name_without_ext = os.path.splitext(filename)[0]
    output_audio_path = os.path.join(AUDIO_DIR, f"{name_without_ext}_dubbed.wav")
    timing_data = []
    regions_to_process = [r for r in speech_regions if r["speech"]]
    logger.info(f"Processing {len(segments)} segments into {len(regions_to_process)} speech regions")
    for idx, seg in enumerate(segments):
        text = seg.get("text", "").strip()
        if not text:
            continue
        orig_start = seg.get("start", 0)
        orig_end = seg.get("end", orig_start + 2)
        orig_duration = orig_end - orig_start
        target_start = orig_start
        target_end = orig_end
        for region in regions_to_process:
            if region["start"] <= orig_start <= region["end"]:
                target_start = region["start"]
                target_end = region["end"]
                break
        target_duration = target_end - target_start
        if seg.get("keep_original", False):
            logger.info(f"Segment {idx+1}: Using original audio (counting)")
            start_sample = int(target_start * sample_rate)
            end_sample = int(target_end * sample_rate)
            if end_sample > len(original_audio):
                end_sample = len(original_audio)
            if start_sample < len(original_audio) and end_sample <= len(original_audio):
                original_segment = original_audio[start_sample:end_sample]
                target_samples = int(target_duration * sample_rate)
                if len(original_segment) < target_samples:
                    silence = np.zeros(target_samples - len(original_segment), dtype=np.float32)
                    original_segment = np.concatenate([original_segment, silence])
                elif len(original_segment) > target_samples:
                    original_segment = original_segment[:target_samples]
                if start_sample + len(original_segment) <= len(final_audio):
                    final_audio[start_sample:start_sample + len(original_segment)] = original_segment
            timing_data.append({"text": text, "start": float(target_start), "end": float(target_end), "type": "original"})
            continue
        logger.info(f"Segment {idx+1}: {orig_duration:.2f}s -> {target_duration:.2f}s")
        try:
            with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp_file:
                tmp_path = tmp_file.name
            asyncio.run(_synthesize_segment(text, voice, tmp_path))
            wav_data, _ = librosa.load(tmp_path, sr=sample_rate, mono=True)
            os.unlink(tmp_path)
            target_samples = int(target_duration * sample_rate)
            if len(wav_data) > 0 and target_samples > 0:
                rate = len(wav_data) / target_samples
                if 0.5 <= rate <= 2.0:
                    wav_data = librosa.effects.time_stretch(wav_data, rate=rate)
                elif rate > 2.0:
                    wav_data = librosa.effects.time_stretch(wav_data, rate=2.0)
                elif rate < 0.5:
                    wav_data = librosa.effects.time_stretch(wav_data, rate=0.5)
            if len(wav_data) < target_samples:
                silence = np.zeros(target_samples - len(wav_data), dtype=np.float32)
                wav_data = np.concatenate([wav_data, silence])
            elif len(wav_data) > target_samples:
                wav_data = wav_data[:target_samples]
            start_sample = int(target_start * sample_rate)
            end_sample = start_sample + len(wav_data)
            if end_sample > len(final_audio):
                padding = np.zeros(end_sample - len(final_audio), dtype=np.float32)
                final_audio = np.concatenate([final_audio, padding])
            final_audio[start_sample:end_sample] = wav_data.astype(np.float32)
            timing_data.append({"text": text, "start": float(target_start), "end": float(target_end), "type": "dubbed"})
        except Exception as e:
            logger.warning(f"Failed TTS: {e}")
            continue
        if progress_callback:
            progress_callback((idx + 1) / len(segments), f"Generated {idx + 1}/{len(segments)}")
    max_val = np.max(np.abs(final_audio))
    if max_val > 0:
        final_audio = final_audio / max_val * 0.95
    logger.info(f"Saving to: {output_audio_path}")
    sf.write(output_audio_path, final_audio, sample_rate)
    import json
    timing_map_path = os.path.join(AUDIO_DIR, f"{name_without_ext}_timing.json")
    with open(timing_map_path, "w") as f:
        json.dump({"segments": timing_data, "speech_regions": speech_regions}, f)
    return output_audio_path, timing_map_path
'''

with open(PROJECT_DIR / "tts_generation.py", "w") as f:
    f.write(tts_content)

print("Created tts_generation.py")

In [ ]:
# Create video_merger.py
merger_content = '''
import os
import subprocess
from config import OUTPUTS_DIR
from utils import logger

def run_lipsync(video_path: str, dubbed_audio_path: str, timing_map_path: str = None) -> str:
    logger.info("Merging generated Hindi audio with the original video...")
    filename = os.path.basename(video_path)
    name_without_ext = os.path.splitext(filename)[0]
    final_output_path = os.path.join(OUTPUTS_DIR, f"{name_without_ext}_final.mp4")
    try:
        video_duration = None
        audio_duration = None
        try:
            video_dur = subprocess.run(
                f"ffprobe -v error -show_entries format=duration -of default=noprintwrappers=1:nokey=1 \"{video_path}\"",
                shell=True, capture_output=True, text=True, check=True
            ).stdout.strip()
            video_duration = float(video_dur)
        except:
            logger.warning("Could not get video duration")
        try:
            audio_dur = subprocess.run(
                f"ffprobe -v error -show_entries format=duration -of default=noprintwrappers=1:nokey=1 \"{dubbed_audio_path}\"",
                shell=True, capture_output=True, text=True, check=True
            ).stdout.strip()
            audio_duration = float(audio_dur)
        except:
            logger.warning("Could not get audio duration")
        if video_duration and audio_duration:
            logger.info(f"Video: {video_duration:.2f}s, Audio: {audio_duration:.2f}s")
        temp_video = video_path
        if video_duration and audio_duration and audio_duration > video_duration:
            try:
                logger.info(f"Extending video from {video_duration:.2f}s to {audio_duration:.2f}s")
                temp_extended = video_path.replace(".mp4", "_extended.mp4")
                subprocess.run(
                    f'ffmpeg -y -i "{video_path}" -f lavfi -i color=c=black:s=640x360:r=25 -filter_complex "[0:v]loop=999:1:0,setpts=N/FRAME_RATE/TB[a]" -map "[a]" -t {audio_duration} -c:v libx264 -preset fast -crf 23 "{temp_extended}"',
                    shell=True, capture_output=True, check=True
                )
                temp_video = temp_extended
            except Exception as e:
                logger.warning(f"Could not extend video: {e}")
        subprocess.run(
            f'ffmpeg -y -i "{temp_video}" -i "{dubbed_audio_path}" -map 0:v:0 -map 1:a:0 -c:v copy -c:a aac -b:a 192k -ar 44100 -ac 2 "{final_output_path}"',
            shell=True, capture_output=True, check=True
        )
        if temp_video != video_path and os.path.exists(temp_video):
            os.remove(temp_video)
        logger.info(f"Video merged successfully! Final dubbed video saved to: {final_output_path}")
        return final_output_path
    except Exception as e:
        logger.error(f"Failed during video assembly. Error: {e}")
        raise e
'''

with open(PROJECT_DIR / "video_merger.py", "w") as f:
    f.write(merger_content)

print("Created video_merger.py")

In [ ]:
# Create downloader.py
dl_content = '''
import os
import yt_dlp
from config import VIDEOS_DIR
from utils import logger

def download_youtube_video(url: str) -> str:
    logger.info(f"Downloading video from YouTube: {url}")
    ydl_opts = {
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
        "outtmpl": str(VIDEOS_DIR / "%(title)s.%(ext)s"),
        "quiet": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        filename = ydl.prepare_filename(info)
    logger.info(f"Video downloaded: {filename}")
    return filename
'''

with open(PROJECT_DIR / "downloader.py", "w") as f:
    f.write(dl_content)

print("Created downloader.py")

In [ ]:
# Create main.py
main_content = '''
import os
import time
import ffmpeg
from pathlib import Path

from utils import logger, initialize_project_environment
from downloader import download_youtube_video
from audio_processing import extract_audio
from transcription import transcribe_audio
from translation import translate_segments
from tts_generation import generate_dubbed_audio
from video_merger import run_lipsync

def get_video_duration(video_path: str) -> float:
    try:
        probe = ffmpeg.probe(video_path)
        return float(probe["format"]["duration"])
    except Exception as e:
        logger.warning(f"Failed to extract video duration: {e}")
        return 1000.0

def run_pipeline(video_source: str):
    logger.info("========== Starting Pipeline ==========")
    initialize_project_environment()
    is_url = video_source.startswith("http://") or video_source.startswith("https://")
    
    if is_url:
        video_path = download_youtube_video(video_source)
    else:
        video_path = video_source
        if not os.path.exists(video_path):
            raise FileNotFoundError(f"Video file not found: {video_path}")
    
    video_duration = get_video_duration(video_path)
    
    logger.info("Extracting audio...")
    audio_path = extract_audio(video_path)
    
    logger.info("Transcribing audio...")
    english_segments = transcribe_audio(audio_path)
    
    logger.info("Translating to Hindi...")
    hindi_segments = translate_segments(english_segments)
    
    logger.info("Generating Hindi audio...")
    dubbed_audio_path, timing_map_path = generate_dubbed_audio(
        hindi_segments, audio_path, video_duration
    )
    
    logger.info("Merging video and audio...")
    final_video_path = run_lipsync(video_path, dubbed_audio_path, timing_map_path)
    
    logger.info("========== Pipeline Finished ==========")
    logger.info(f"Final output: {final_video_path}")
    return final_video_path
'''

with open(PROJECT_DIR / "main.py", "w") as f:
    f.write(main_content)

print("Created main.py")

## 4. Run Dubbing Pipeline

In [ ]:
# Import and run
import sys
sys.path.insert(0, str(PROJECT_DIR))

# Option 1: From YouTube URL
# video_url = "https://www.youtube.com/watch?v=EXAMPLE"
# output = run_pipeline(video_url)

# Option 2: From local file (upload to videos/ folder)
# output = run_pipeline("/content/voice_dubbing/videos/myvideo.mp4")

print("Setup complete! Uncomment and run one of the options above.")

## 5. Download Output

In [ ]:
# List output files
import os
for f in os.listdir(PROJECT_DIR / "outputs"):
    print(f)

# Download from Colab
# from google.colab import files
# files.download(str(PROJECT_DIR / "outputs" / "your_video_final.mp4"))